Title: download_v2.ipynb

Purpose: Download CMIP6 Data and create a list of used runs

Author: Onno Nennecke on 15.03.2025 Modified: 16.06.2025

Output data:

- CMIP6 data lies here: '/climca/data/'
- List of runs lies here: '/home/onennecke/CMIP_models/CMIP6_runs.csv'

#### Load packages

In [1]:
import esmvalcore.esgf
import numpy as np
import pandas as pd
import os
import glob

In [ ]:
ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'CESM2-WACCM', 'CNRM-CM6-1', 'CNRM-CM6-1-HR', 'CNRM-ESM2-1', 'EC-Earth3',
        'EC-Earth3-Veg', 'GFDL-CM4', 'GFDL-ESM4', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM', 'MPI-ESM1-2-HR', 
        'MRI-ESM2-0', 'KACE-1-0-G', 'TaiESM1', 'UKESM1-0-LL']

In [ ]:
for ESM in ESMs:
    fls = esmvalcore.esgf.find_files(
        project='CMIP6',
        mip='day',
        short_name='tas',
        dataset=ESM,
        exp='ssp370',
        ensemble='*',
    )  
    ensemble_names = np.unique([fl.__dict__['facets']['ensemble'] for fl in fls])

    for ensemble in ensemble_names:
        download = True
        files_to_download = []
        for var in ['tas','tasmax','rsds','sfcWind', 'psl']: # 'zg'
            fls = esmvalcore.esgf.find_files(
                project='CMIP6',
                mip='day',
                short_name=var,
                dataset=ESM,
                exp='ssp370',
                ensemble=ensemble,
            )  
            useful_files_for_variable = []
            for fl in fls:
                useful_files_for_variable.append(fl)
                if int(fl.name.split('-')[-1][:4]) >= 2024:
                    break
            files_to_download += useful_files_for_variable

            if len(useful_files_for_variable) == 0:
                download = False
                break


        if download:
            print('Download', ESM, ensemble)
            print(ESM,files_to_download)
            esmvalcore.esgf.download(files_to_download, '/climca/data/', n_jobs=4)


Download ACCESS-CM2 r1i1p1f1
ACCESS-CM2 [ESGFFile:CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day/tas/gn/v20191108/tas_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20641231.nc on hosts ['esgf-data1.llnl.gov', 'esgf-data04.diasjp.net', 'esgf.ceda.ac.uk', 'esgf.nci.org.au', 'esgf3.dkrz.de'], ESGFFile:CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day/tasmax/gn/v20191108/tasmax_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20641231.nc on hosts ['esgf-data1.llnl.gov', 'esgf-data04.diasjp.net', 'esgf.ceda.ac.uk', 'esgf.nci.org.au', 'esgf3.dkrz.de'], ESGFFile:CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day/rsds/gn/v20191108/rsds_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20641231.nc on hosts ['esgf-data04.diasjp.net', 'esgf-data1.llnl.gov', 'esgf.ceda.ac.uk', 'esgf.nci.org.au', 'esgf3.dkrz.de'], ESGFFile:CMIP6/ScenarioMIP/CSIRO-ARCCSS/ACCESS-CM2/ssp370/r1i1p1f1/day/sfcWind/gn/v20191108/sfcWind_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20641231.nc on hosts [

PermissionError: [Errno 13] Permission denied: '/climca/data/CMIP6/ScenarioMIP/EC-Earth-Consortium/EC-Earth3/ssp370/r11i1p1f1/day/rsds/gr/v20200201/rsds_day_EC-Earth3_ssp370_r11i1p1f1_gr_20190101-20191231.nc.okvz_t_v'

----

##  Make a list of all usable runs

### Define used models

In [2]:
# Load climate data

MIP = 'ScenarioMIP' # CMIP

Institution = ['CSIRO-ARCCSS', 'BCC', 'NCAR', 'EC-Earth-Consortium', 'NOAA-GFDL', 'NIMS-KMA', 'DKRZ', 'MRI', 'AS-RCEC', 'MOHC', 'NIMS-KMA']
ESMs = ['ACCESS-CM2', 'BCC-CSM2-MR', 'CESM2', 'EC-Earth3', 'GFDL-ESM4', 'KACE-1-0-G', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'TaiESM1', 'UKESM1-0-LL', 'UKESM1-0-LL']

scenario = 'ssp370'

time_res = 'day'
variables = ['sfcWind', 'rsds', 'tas', 'tasmax'] # List of variables
variables = ['sfcWind', 'rsds', 'tas', 'tasmax', 'psl'] # List of variables
grid_def = '*'
version = '*'

In [3]:
# Make an empty pandas dataframe with ESM, Institution and runs as columns
# df = pd.DataFrame(columns=['ESM', 'Institution', 'run', 'variable'])
df = pd.DataFrame(columns=['ESM', 'Institution', 'run'])

In [4]:
# Takes about 
for ESM, Inst in zip(ESMs, Institution):
    print('ESM: ', ESM)
    path = f'/climca/data/CMIP6/{MIP}/{Inst}/{ESM}/{scenario}/'
    matching_dirs = glob.glob(path)
    runs = os.listdir(matching_dirs[0])
    print('Runs: ', runs)
    
    esm_run_datasets = []
    
    for run in runs:
        ds_list = [] # List to hold individual datasets (one for each variable)

        print('Run: ', run, 'Number: ', runs.index(run) + 1, 'of ', len(runs))
        run_path = os.path.join(matching_dirs[0], run, 'day/')
        
        # Check if all required folders (in `variables`) exist
        missing_folders = [var for var in variables if not os.path.isdir(os.path.join(run_path, var))]
        
        if missing_folders:
            print(f"Missing folders in {run_path}: {missing_folders}")
            continue
        
        missing_data = [
            var for var in variables 
            if not os.path.isdir(os.path.join(run_path, var)) or 
               not any(
                   os.path.isfile(f) 
                   for f in glob.glob(os.path.join(run_path, var, '*', '*', '*'))  # glob pattern to match files two levels down
               )
        ]
        
        if missing_data:
            print(f"Missing data in {run_path}: {missing_data}")
            continue
        df.loc[len(df)] = [ESM, Inst, run] # Without variables
        # for var in variables:
        #     df.loc[len(df)] = [ESM, Inst, run, var]

# Save the dataframe to a csv file
df.to_csv('/home/onennecke/CMIP_models/CMIP6_runs.csv', index=False)

ESM:  ACCESS-CM2
Runs:  ['r4i1p1f1', 'r5i1p1f1', 'r1i1p1f1']
Run:  r4i1p1f1 Number:  1 of  3
Run:  r5i1p1f1 Number:  2 of  3
Run:  r1i1p1f1 Number:  3 of  3
ESM:  BCC-CSM2-MR
Runs:  ['r1i1p1f1']
Run:  r1i1p1f1 Number:  1 of  1
ESM:  CESM2
Runs:  ['r4i1p1f1', 'r10i1p1f1', 'r11i1p1f1']
Run:  r4i1p1f1 Number:  1 of  3
Run:  r10i1p1f1 Number:  2 of  3
Run:  r11i1p1f1 Number:  3 of  3
ESM:  EC-Earth3
Runs:  ['r149i1p1f1', 'r6i1p1f1', 'r4i1p1f1', 'r148i1p1f1', 'r105i1p1f1', 'r9i1p1f1', 'r134i1p1f1', 'r141i1p1f1', 'r146i1p1f1', 'r15i1p1f1', 'r112i1p1f1', 'r117i1p1f1', 'r125i1p1f1', 'r113i1p1f1', 'r106i1p1f1', 'r138i1p1f1', 'r5i1p1f1', 'r137i1p1f1', 'r11i1p1f1', 'r145i1p1f1', 'r114i1p1f1', 'r120i1p1f1', 'r128i1p1f1', 'r135i1p1f1', 'r110i1p1f1', 'r129i1p1f1', 'r132i1p1f1', 'r101i1p1f1', 'r124i1p1f1', 'r127i1p1f1', 'r116i1p1f1', 'r131i1p1f1', 'r121i1p1f1', 'r142i1p1f1', 'r102i1p1f1', 'r133i1p1f1', 'r111i1p1f1', 'r140i1p1f1', 'r136i1p1f1', 'r108i1p1f1', 'r130i1p1f1', 'r150i1p1f1', 'r104i1p1f1', '

In [5]:
# Read the dataframe from the csv file
df = pd.read_csv('/home/onennecke/CMIP_models/CMIP6_runs.csv')
df

,ESM,Institution,run
0,ACCESS-CM2,CSIRO-ARCCSS,r4i1p1f1
1,ACCESS-CM2,CSIRO-ARCCSS,r5i1p1f1
2,ACCESS-CM2,CSIRO-ARCCSS,r1i1p1f1
3,BCC-CSM2-MR,BCC,r1i1p1f1
4,CESM2,NCAR,r4i1p1f1
...,...,...,...
94,UKESM1-0-LL,MOHC,r3i1p1f2
95,UKESM1-0-LL,MOHC,r8i1p1f2
96,UKESM1-0-LL,NIMS-KMA,r15i1p1f2
97,UKESM1-0-LL,NIMS-KMA,r13i1p1f2
